# Week 04 Learning Notebook

This notebook mirrors the daily lesson plan and includes runnable demo code cells.

In [1]:
from datetime import date
import warnings
import numpy as np
import pandas as pd

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.6f}'.format

def _synthetic_price_frame(tickers, start='2018-01-01', periods=900, seed=7):
    idx = pd.bdate_range(start, periods=periods)
    out = pd.DataFrame(index=idx)
    rng = np.random.default_rng(seed)
    for i, ticker in enumerate(tickers):
        drift = 0.0002 + 0.00005 * (i + 1)
        vol = 0.010 + 0.002 * i
        rets = rng.normal(drift, vol, size=len(idx))
        out[ticker] = 100.0 * np.exp(np.cumsum(rets))
    return out

def load_market_prices(tickers, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    tickers = list(tickers)
    if yf is None:
        warnings.warn('yfinance unavailable, using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            if 'Close' in raw.columns.get_level_values(0):
                close = raw['Close']
            else:
                close = raw.xs('Close', axis=1, level=0, drop_level=True)
        else:
            close = raw.rename(columns={raw.columns[0]: tickers[0]})
        close = close.reindex(columns=tickers)
        close = close.dropna(how='all').ffill().dropna()
        if close.empty:
            raise ValueError('empty market data from Yahoo Finance')
        return close
    except Exception as exc:
        warnings.warn(f'Yahoo download failed ({exc}); using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

def load_fred_series(series_id, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    if pdr is None:
        warnings.warn('pandas_datareader unavailable, using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

    try:
        ser = pdr.DataReader(series_id, 'fred', start, end)[series_id]
        ser = ser.ffill().dropna()
        if ser.empty:
            raise ValueError('empty FRED series')
        return ser
    except Exception as exc:
        warnings.warn(f'FRED download failed ({exc}); using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

# Week 04 Day 01: Return decomposition and compounding effects

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Integrate foundational math and data skills into a complete exploratory market analysis.

## Continuity and Handoff
- Previous checkpoint: Week 03 Day 07: Portfolio Project
- Previous lesson file: content/week-03/day-07.md
- Today's deliverable: Implement return decomposition diagnostics in a reusable notebook section.
- Next handoff target: Week 04 Day 02: Risk distribution diagnostics
- Next lesson file: content/week-04/day-02.md

## Theory Concepts

### Concept 1: Arithmetic vs geometric interpretation
Arithmetic vs geometric interpretation is a core part of 'Return/risk analytics and exploratory capstone'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Volatility drag
Volatility drag is a core part of 'Return/risk analytics and exploratory capstone'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Regime-dependent performance
Regime-dependent performance is a core part of 'Return/risk analytics and exploratory capstone'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Simple Return
$$
r_t = \frac{P_t - P_{t-1}}{P_{t-1}}
$$
**Plain-English interpretation:** Normalize raw price moves.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 2: Log Return
$$
\ell_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$
**Plain-English interpretation:** Additive return representation.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 3: Annualized Volatility
$$
\sigma_{ann}=\sqrt{252}\,Std(r_t)
$$
**Plain-English interpretation:** Scale daily uncertainty to annual horizon.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $R_f$ | Risk-free rate | annualized decimal | 0.03 |
| $TO_t$ | Portfolio turnover | fraction of portfolio | 0.12 |

## Real Trading Example
- Instruments: SPY, QQQ, AAPL
- Macro overlay (FRED): DGS10, UNRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-02 to 2020-04, 2022-01 to 2022-10
- Scenario context: inflation surprise and policy-rate repricing
- Day objective: Compare cumulative growth under identical mean returns but different volatilities.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Return decomposition and compounding effects'.

## Step-by-Step Solved Problems
### Solved Problem 1: Compute simple return
Given:
- Price moves from $102.00 to $103.22.
Solution:
1. $r_t=\frac{P_t-P_{t-1}}{P_{t-1}}$.
2. r_t = (103.224-102.000)/102.000 = 0.012000.
Final answer: Simple return = 1.20%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Annualize volatility
Given:
- Daily volatility estimate is 0.0112.
Solution:
1. $\sigma_{ann}=\sqrt{252}\cdot\sigma_d$.
2. sigma_ann = sqrt(252)*0.0112 = 0.1778.
Final answer: Annualized volatility = 17.78%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute Sharpe ratio
Given:
- Annual return is 14.0%, risk-free rate is 3.0%.
- Use volatility 17.78%.
Solution:
1. $S=\frac{R_{ann}-R_f}{\sigma_{ann}}$.
2. S = (0.14-0.03)/0.1778 = 0.6187.
Final answer: Sharpe ratio = 0.62.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement return decomposition diagnostics in a reusable notebook section.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
prices = load_prices(['SPY', 'QQQ', 'AAPL'])
returns = prices.pct_change().dropna()
metrics = compute_risk_metrics(returns)
assert metrics['max_drawdown'] <= 0
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 04 Day 01): Explain Simple Return and define every symbol clearly for an inflation surprise week with simultaneous equity and rate volatility.
   - Model answer: "I use Simple Return as a decision bridge from market observations to position sizing. The formula is $r_t = \frac{P_t - P_{t-1}}{P_{t-1}}$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then cap gross exposure when rolling 20-day volatility exceeds backtest 90th percentile. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Production question: Why does 'Return decomposition and compounding effects' matter in live trading systems?
   - Model answer: "Return decomposition and compounding effects matters because reliable notation and units prevent silent compounding and scaling errors in production PnL reporting. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and one production escalation rule.

### Interview Drill
- Prompt: "Walk me through Return decomposition and compounding effects in a PM review after CPI surprise where benchmark drawdown accelerated."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
How does volatility drag influence long-horizon planning?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 04 Day 01 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [2]:
# Foundation demo: real prices + macro overlay with fallback
np.random.seed(401)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()
macro = load_fred_series('DGS10', start='2018-01-01').rename('dgs10')
joined = returns.join(macro, how='inner').dropna()

growth = float((1 + joined['SPY']).prod())
ann_return = float((1 + joined['SPY']).prod() ** (252 / joined.shape[0]) - 1)
ann_vol = float(joined['SPY'].std() * np.sqrt(252))
drawdown = float((prices['SPY'] / prices['SPY'].cummax() - 1).min())
rate_corr = float(joined['SPY'].corr(joined['dgs10']))

{
    'growth_multiple': growth,
    'annualized_return': ann_return,
    'annualized_volatility': ann_vol,
    'max_drawdown': drawdown,
    'spy_rate_corr': rate_corr,
}


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_46794/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'growth_multiple': 3.001916748261975,
 'annualized_return': 0.14223478242587606,
 'annualized_volatility': 0.19307148884826125,
 'max_drawdown': -0.33717259886422934,
 'spy_rate_corr': 0.008052798822480355}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [3]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(9401)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Return decomposition and compounding effects',
    'week': 4,
    'day': 1,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Return decomposition and compounding effects',
 'week': 4,
 'day': 1,
 'observe_annual_return': 0.14923384092579295,
 'observe_annual_vol': 0.20501282184020983,
 'reason_sharpe_proxy': 0.581592116315172,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 04 Day 01 Quiz

Topic: **Return decomposition and compounding effects**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [4]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 105.000
price_t = 105.892
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages under inflation surprise week.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  cap exposure if rolling volatility breaches training 90th percentile.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Return decomposition and compounding effects')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.008495)
print('  gross_return_expected  =', 1.008495)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages under inflation surprise week.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 105.0
  P_t    : 105.892
  r_t    : 0.008495 => 0.85%
  1+r_t  : 1.008495

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  cap exposure if rolling volatility breaches training 90th percentile.

Interview Question 3 (model answer):
  Topic: Return decomposition and compounding effects
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Numeric verification:
  simple_return_expected = 0.008495
  gross_return_expected  = 1.008495


# Week 04 Day 02: Risk distribution diagnostics

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Integrate foundational math and data skills into a complete exploratory market analysis.

## Continuity and Handoff
- Previous checkpoint: Week 04 Day 01: Return decomposition and compounding effects
- Previous lesson file: content/week-04/day-01.md
- Today's deliverable: Build a downside-risk summary table including semi-variance.
- Next handoff target: Week 04 Day 03: Correlation structure and diversification
- Next lesson file: content/week-04/day-03.md

## Theory Concepts

### Concept 1: Skewness and kurtosis in return data
Skewness and kurtosis in return data is a core part of 'Return/risk analytics and exploratory capstone'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Tail event frequency
Tail event frequency is a core part of 'Return/risk analytics and exploratory capstone'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Downside-focused metrics
Downside-focused metrics is a core part of 'Return/risk analytics and exploratory capstone'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Log Return
$$
\ell_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$
**Plain-English interpretation:** Additive return representation.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 2: Annualized Volatility
$$
\sigma_{ann}=\sqrt{252}\,Std(r_t)
$$
**Plain-English interpretation:** Scale daily uncertainty to annual horizon.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 3: Sharpe Ratio
$$
S=\frac{R_{ann}-R_f}{\sigma_{ann}}
$$
**Plain-English interpretation:** Risk-adjusted performance score.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $R_f$ | Risk-free rate | annualized decimal | 0.03 |
| $TO_t$ | Portfolio turnover | fraction of portfolio | 0.12 |

## Real Trading Example
- Instruments: SPY, QQQ, AAPL
- Macro overlay (FRED): DGS10, UNRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-02 to 2020-04, 2022-01 to 2022-10
- Scenario context: inflation surprise and policy-rate repricing
- Day objective: Evaluate return histograms and downside risk frequencies for multiple assets.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Risk distribution diagnostics'.

## Step-by-Step Solved Problems
### Solved Problem 1: Compute simple return
Given:
- Price moves from $104.00 to $105.35.
Solution:
1. $r_t=\frac{P_t-P_{t-1}}{P_{t-1}}$.
2. r_t = (105.352-104.000)/104.000 = 0.013000.
Final answer: Simple return = 1.30%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Annualize volatility
Given:
- Daily volatility estimate is 0.0119.
Solution:
1. $\sigma_{ann}=\sqrt{252}\cdot\sigma_d$.
2. sigma_ann = sqrt(252)*0.0119 = 0.1889.
Final answer: Annualized volatility = 18.89%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute Sharpe ratio
Given:
- Annual return is 14.0%, risk-free rate is 3.0%.
- Use volatility 18.89%.
Solution:
1. $S=\frac{R_{ann}-R_f}{\sigma_{ann}}$.
2. S = (0.14-0.03)/0.1889 = 0.5823.
Final answer: Sharpe ratio = 0.58.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Build a downside-risk summary table including semi-variance.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
prices = load_prices(['SPY', 'QQQ', 'AAPL'])
returns = prices.pct_change().dropna()
metrics = compute_risk_metrics(returns)
assert metrics['max_drawdown'] <= 0
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 04 Day 02): Explain Log Return and define every symbol clearly for an inflation surprise week with simultaneous equity and rate volatility.
   - Model answer: "I use Log Return as a decision bridge from market observations to position sizing. The formula is $\ell_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then cap gross exposure when rolling 20-day volatility exceeds backtest 90th percentile. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Production question: Why does 'Risk distribution diagnostics' matter in live trading systems?
   - Model answer: "Risk distribution diagnostics matters because reliable notation and units prevent silent compounding and scaling errors in production PnL reporting. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and one production escalation rule.

### Interview Drill
- Prompt: "Walk me through Risk distribution diagnostics in a PM review after CPI surprise where benchmark drawdown accelerated."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
Why can variance miss path-dependent pain?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 04 Day 02 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [5]:
# Foundation demo: real prices + macro overlay with fallback
np.random.seed(402)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()
macro = load_fred_series('DGS10', start='2018-01-01').rename('dgs10')
joined = returns.join(macro, how='inner').dropna()

growth = float((1 + joined['SPY']).prod())
ann_return = float((1 + joined['SPY']).prod() ** (252 / joined.shape[0]) - 1)
ann_vol = float(joined['SPY'].std() * np.sqrt(252))
drawdown = float((prices['SPY'] / prices['SPY'].cummax() - 1).min())
rate_corr = float(joined['SPY'].corr(joined['dgs10']))

{
    'growth_multiple': growth,
    'annualized_return': ann_return,
    'annualized_volatility': ann_vol,
    'max_drawdown': drawdown,
    'spy_rate_corr': rate_corr,
}


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_46794/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'growth_multiple': 3.001916748261975,
 'annualized_return': 0.14223478242587606,
 'annualized_volatility': 0.19307148884826125,
 'max_drawdown': -0.33717259886422934,
 'spy_rate_corr': 0.008052798822480355}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [6]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(9402)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Risk distribution diagnostics',
    'week': 4,
    'day': 2,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Risk distribution diagnostics',
 'week': 4,
 'day': 2,
 'observe_annual_return': 0.14923384092579295,
 'observe_annual_vol': 0.20501282184020983,
 'reason_sharpe_proxy': 0.581592116315172,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 04 Day 02 Quiz

Topic: **Risk distribution diagnostics**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [7]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 106.000
price_t = 106.954
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages under inflation surprise week.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  cap exposure if rolling volatility breaches training 90th percentile.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Risk distribution diagnostics')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.009000)
print('  gross_return_expected  =', 1.009000)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages under inflation surprise week.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 106.0
  P_t    : 106.954
  r_t    : 0.009 => 0.90%
  1+r_t  : 1.009

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  cap exposure if rolling volatility breaches training 90th percentile.

Interview Question 3 (model answer):
  Topic: Risk distribution diagnostics
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Numeric verification:
  simple_return_expected = 0.009
  gross_return_expected  = 1.009


# Week 04 Day 03: Correlation structure and diversification

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Integrate foundational math and data skills into a complete exploratory market analysis.

## Continuity and Handoff
- Previous checkpoint: Week 04 Day 02: Risk distribution diagnostics
- Previous lesson file: content/week-04/day-02.md
- Today's deliverable: Generate rolling-correlation heatmaps over time windows.
- Next handoff target: Week 04 Day 04: Exploratory visualization for strategy intuition
- Next lesson file: content/week-04/day-04.md

## Theory Concepts

### Concept 1: Rolling correlation behavior
Rolling correlation behavior is a core part of 'Return/risk analytics and exploratory capstone'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Diversification under stress
Diversification under stress is a core part of 'Return/risk analytics and exploratory capstone'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Clustered risk exposures
Clustered risk exposures is a core part of 'Return/risk analytics and exploratory capstone'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Annualized Volatility
$$
\sigma_{ann}=\sqrt{252}\,Std(r_t)
$$
**Plain-English interpretation:** Scale daily uncertainty to annual horizon.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 2: Sharpe Ratio
$$
S=\frac{R_{ann}-R_f}{\sigma_{ann}}
$$
**Plain-English interpretation:** Risk-adjusted performance score.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 3: Turnover
$$
TO_t=\frac{1}{2}\sum_i|w_{i,t}-w_{i,t-1}|
$$
**Plain-English interpretation:** Execution intensity proxy.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $R_f$ | Risk-free rate | annualized decimal | 0.03 |
| $TO_t$ | Portfolio turnover | fraction of portfolio | 0.12 |

## Real Trading Example
- Instruments: SPY, QQQ, AAPL
- Macro overlay (FRED): DGS10, UNRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-02 to 2020-04, 2022-01 to 2022-10
- Scenario context: inflation surprise and policy-rate repricing
- Day objective: Track rolling correlations during synthetic stress intervals.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Correlation structure and diversification'.

## Step-by-Step Solved Problems
### Solved Problem 1: Compute simple return
Given:
- Price moves from $106.00 to $107.48.
Solution:
1. $r_t=\frac{P_t-P_{t-1}}{P_{t-1}}$.
2. r_t = (107.484-106.000)/106.000 = 0.014000.
Final answer: Simple return = 1.40%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Annualize volatility
Given:
- Daily volatility estimate is 0.0126.
Solution:
1. $\sigma_{ann}=\sqrt{252}\cdot\sigma_d$.
2. sigma_ann = sqrt(252)*0.0126 = 0.2000.
Final answer: Annualized volatility = 20.00%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute Sharpe ratio
Given:
- Annual return is 14.0%, risk-free rate is 3.0%.
- Use volatility 20.00%.
Solution:
1. $S=\frac{R_{ann}-R_f}{\sigma_{ann}}$.
2. S = (0.14-0.03)/0.2000 = 0.5499.
Final answer: Sharpe ratio = 0.55.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Generate rolling-correlation heatmaps over time windows.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
prices = load_prices(['SPY', 'QQQ', 'AAPL'])
returns = prices.pct_change().dropna()
metrics = compute_risk_metrics(returns)
assert metrics['max_drawdown'] <= 0
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 04 Day 03): Explain Annualized Volatility and define every symbol clearly for an inflation surprise week with simultaneous equity and rate volatility.
   - Model answer: "I use Annualized Volatility as a decision bridge from market observations to position sizing. The formula is $\sigma_{ann}=\sqrt{252}\,Std(r_t)$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then cap gross exposure when rolling 20-day volatility exceeds backtest 90th percentile. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Production question: Why does 'Correlation structure and diversification' matter in live trading systems?
   - Model answer: "Correlation structure and diversification matters because reliable notation and units prevent silent compounding and scaling errors in production PnL reporting. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and one production escalation rule.

### Interview Drill
- Prompt: "Walk me through Correlation structure and diversification in a PM review after CPI surprise where benchmark drawdown accelerated."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
When does diversification break down?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 04 Day 03 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [8]:
# Foundation demo: real prices + macro overlay with fallback
np.random.seed(403)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()
macro = load_fred_series('DGS10', start='2018-01-01').rename('dgs10')
joined = returns.join(macro, how='inner').dropna()

growth = float((1 + joined['SPY']).prod())
ann_return = float((1 + joined['SPY']).prod() ** (252 / joined.shape[0]) - 1)
ann_vol = float(joined['SPY'].std() * np.sqrt(252))
drawdown = float((prices['SPY'] / prices['SPY'].cummax() - 1).min())
rate_corr = float(joined['SPY'].corr(joined['dgs10']))

{
    'growth_multiple': growth,
    'annualized_return': ann_return,
    'annualized_volatility': ann_vol,
    'max_drawdown': drawdown,
    'spy_rate_corr': rate_corr,
}


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_46794/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'growth_multiple': 3.001916748261975,
 'annualized_return': 0.14223478242587606,
 'annualized_volatility': 0.19307148884826125,
 'max_drawdown': -0.33717259886422934,
 'spy_rate_corr': 0.008052798822480355}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [9]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(9403)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Correlation structure and diversification',
    'week': 4,
    'day': 3,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Correlation structure and diversification',
 'week': 4,
 'day': 3,
 'observe_annual_return': 0.14923384092579295,
 'observe_annual_vol': 0.20501282184020983,
 'reason_sharpe_proxy': 0.581592116315172,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 04 Day 03 Quiz

Topic: **Correlation structure and diversification**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [10]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 107.000
price_t = 108.017
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages under inflation surprise week.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  cap exposure if rolling volatility breaches training 90th percentile.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Correlation structure and diversification')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.009505)
print('  gross_return_expected  =', 1.009505)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages under inflation surprise week.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 107.0
  P_t    : 108.017
  r_t    : 0.009505 => 0.95%
  1+r_t  : 1.009505

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  cap exposure if rolling volatility breaches training 90th percentile.

Interview Question 3 (model answer):
  Topic: Correlation structure and diversification
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Numeric verification:
  simple_return_expected = 0.009505
  gross_return_expected  = 1.009505


# Week 04 Day 04: Exploratory visualization for strategy intuition

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Integrate foundational math and data skills into a complete exploratory market analysis.

## Continuity and Handoff
- Previous checkpoint: Week 04 Day 03: Correlation structure and diversification
- Previous lesson file: content/week-04/day-03.md
- Today's deliverable: Produce clean charts with labeled assumptions and caveats.
- Next handoff target: Week 04 Day 05: Capstone design and reporting
- Next lesson file: content/week-04/day-05.md

## Theory Concepts

### Concept 1: Narrative charting
Narrative charting is a core part of 'Return/risk analytics and exploratory capstone'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Signal sanity checks
Signal sanity checks is a core part of 'Return/risk analytics and exploratory capstone'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Interpretability of exploratory outputs
Interpretability of exploratory outputs is a core part of 'Return/risk analytics and exploratory capstone'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Sharpe Ratio
$$
S=\frac{R_{ann}-R_f}{\sigma_{ann}}
$$
**Plain-English interpretation:** Risk-adjusted performance score.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 2: Turnover
$$
TO_t=\frac{1}{2}\sum_i|w_{i,t}-w_{i,t-1}|
$$
**Plain-English interpretation:** Execution intensity proxy.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 3: Simple Return
$$
r_t = \frac{P_t - P_{t-1}}{P_{t-1}}
$$
**Plain-English interpretation:** Normalize raw price moves.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $R_f$ | Risk-free rate | annualized decimal | 0.03 |
| $TO_t$ | Portfolio turnover | fraction of portfolio | 0.12 |

## Real Trading Example
- Instruments: SPY, QQQ, AAPL
- Macro overlay (FRED): DGS10, UNRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-02 to 2020-04, 2022-01 to 2022-10
- Scenario context: inflation surprise and policy-rate repricing
- Day objective: Build a concise dashboard panel of return, drawdown, and volatility views.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Exploratory visualization for strategy intuition'.

## Step-by-Step Solved Problems
### Solved Problem 1: Compute simple return
Given:
- Price moves from $108.00 to $109.62.
Solution:
1. $r_t=\frac{P_t-P_{t-1}}{P_{t-1}}$.
2. r_t = (109.620-108.000)/108.000 = 0.015000.
Final answer: Simple return = 1.50%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Annualize volatility
Given:
- Daily volatility estimate is 0.0133.
Solution:
1. $\sigma_{ann}=\sqrt{252}\cdot\sigma_d$.
2. sigma_ann = sqrt(252)*0.0133 = 0.2111.
Final answer: Annualized volatility = 21.11%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute Sharpe ratio
Given:
- Annual return is 14.0%, risk-free rate is 3.0%.
- Use volatility 21.11%.
Solution:
1. $S=\frac{R_{ann}-R_f}{\sigma_{ann}}$.
2. S = (0.14-0.03)/0.2111 = 0.5210.
Final answer: Sharpe ratio = 0.52.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Produce clean charts with labeled assumptions and caveats.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
prices = load_prices(['SPY', 'QQQ', 'AAPL'])
returns = prices.pct_change().dropna()
metrics = compute_risk_metrics(returns)
assert metrics['max_drawdown'] <= 0
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 04 Day 04): Explain Sharpe Ratio and define every symbol clearly for an inflation surprise week with simultaneous equity and rate volatility.
   - Model answer: "I use Sharpe Ratio as a decision bridge from market observations to position sizing. The formula is $S=\frac{R_{ann}-R_f}{\sigma_{ann}}$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then cap gross exposure when rolling 20-day volatility exceeds backtest 90th percentile. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Production question: Why does 'Exploratory visualization for strategy intuition' matter in live trading systems?
   - Model answer: "Exploratory visualization for strategy intuition matters because reliable notation and units prevent silent compounding and scaling errors in production PnL reporting. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and one production escalation rule.

### Interview Drill
- Prompt: "Walk me through Exploratory visualization for strategy intuition in a PM review after CPI surprise where benchmark drawdown accelerated."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
How do visuals bias interpretation if scales are inconsistent?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 04 Day 04 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [11]:
# Foundation demo: real prices + macro overlay with fallback
np.random.seed(404)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()
macro = load_fred_series('DGS10', start='2018-01-01').rename('dgs10')
joined = returns.join(macro, how='inner').dropna()

growth = float((1 + joined['SPY']).prod())
ann_return = float((1 + joined['SPY']).prod() ** (252 / joined.shape[0]) - 1)
ann_vol = float(joined['SPY'].std() * np.sqrt(252))
drawdown = float((prices['SPY'] / prices['SPY'].cummax() - 1).min())
rate_corr = float(joined['SPY'].corr(joined['dgs10']))

{
    'growth_multiple': growth,
    'annualized_return': ann_return,
    'annualized_volatility': ann_vol,
    'max_drawdown': drawdown,
    'spy_rate_corr': rate_corr,
}


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_46794/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'growth_multiple': 3.001916748261975,
 'annualized_return': 0.14223478242587606,
 'annualized_volatility': 0.19307148884826125,
 'max_drawdown': -0.33717259886422934,
 'spy_rate_corr': 0.008052798822480355}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [12]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(9404)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Exploratory visualization for strategy intuition',
    'week': 4,
    'day': 4,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Exploratory visualization for strategy intuition',
 'week': 4,
 'day': 4,
 'observe_annual_return': 0.14923384092579295,
 'observe_annual_vol': 0.20501282184020983,
 'reason_sharpe_proxy': 0.581592116315172,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 04 Day 04 Quiz

Topic: **Exploratory visualization for strategy intuition**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [13]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 108.000
price_t = 109.080
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages under inflation surprise week.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  cap exposure if rolling volatility breaches training 90th percentile.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Exploratory visualization for strategy intuition')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.010000)
print('  gross_return_expected  =', 1.010000)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages under inflation surprise week.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 108.0
  P_t    : 109.08
  r_t    : 0.01 => 1.00%
  1+r_t  : 1.01

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  cap exposure if rolling volatility breaches training 90th percentile.

Interview Question 3 (model answer):
  Topic: Exploratory visualization for strategy intuition
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Numeric verification:
  simple_return_expected = 0.01
  gross_return_expected  = 1.01


# Week 04 Day 05: Capstone design and reporting

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Integrate foundational math and data skills into a complete exploratory market analysis.

## Continuity and Handoff
- Previous checkpoint: Week 04 Day 04: Exploratory visualization for strategy intuition
- Previous lesson file: content/week-04/day-04.md
- Today's deliverable: Create a report template with metric tables and key plots.
- Next handoff target: Week 04 Day 06: Revision Sprint
- Next lesson file: content/week-04/day-06.md

## Theory Concepts

### Concept 1: Problem framing for quant research
Problem framing for quant research is a core part of 'Return/risk analytics and exploratory capstone'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Assumption logging
Assumption logging is a core part of 'Return/risk analytics and exploratory capstone'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Communicating findings with uncertainty
Communicating findings with uncertainty is a core part of 'Return/risk analytics and exploratory capstone'. Start with notation discipline: define prices, returns, percentages, and all symbols before doing any arithmetic. Then focus on clean data assumptions and stable mathematical transformations by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Turnover
$$
TO_t=\frac{1}{2}\sum_i|w_{i,t}-w_{i,t-1}|
$$
**Plain-English interpretation:** Execution intensity proxy.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 2: Simple Return
$$
r_t = \frac{P_t - P_{t-1}}{P_{t-1}}
$$
**Plain-English interpretation:** Normalize raw price moves.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

### Formula 3: Log Return
$$
\ell_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$
**Plain-English interpretation:** Additive return representation.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Compute this on SPY and QQQ daily closes, then compare how one volatile day changes the metric.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $R_f$ | Risk-free rate | annualized decimal | 0.03 |
| $TO_t$ | Portfolio turnover | fraction of portfolio | 0.12 |

## Real Trading Example
- Instruments: SPY, QQQ, AAPL
- Macro overlay (FRED): DGS10, UNRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-02 to 2020-04, 2022-01 to 2022-10
- Scenario context: inflation surprise and policy-rate repricing
- Day objective: Draft a one-page executive summary from your exploratory analysis.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Capstone design and reporting'.

## Step-by-Step Solved Problems
### Solved Problem 1: Compute simple return
Given:
- Price moves from $110.00 to $111.76.
Solution:
1. $r_t=\frac{P_t-P_{t-1}}{P_{t-1}}$.
2. r_t = (111.760-110.000)/110.000 = 0.016000.
Final answer: Simple return = 1.60%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Annualize volatility
Given:
- Daily volatility estimate is 0.0140.
Solution:
1. $\sigma_{ann}=\sqrt{252}\cdot\sigma_d$.
2. sigma_ann = sqrt(252)*0.0140 = 0.2222.
Final answer: Annualized volatility = 22.22%.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Compute Sharpe ratio
Given:
- Annual return is 14.0%, risk-free rate is 3.0%.
- Use volatility 22.22%.
Solution:
1. $S=\frac{R_{ann}-R_f}{\sigma_{ann}}$.
2. S = (0.14-0.03)/0.2222 = 0.4950.
Final answer: Sharpe ratio = 0.49.
Common trap: Confusing percentage return with absolute dollar change or forgetting that returns compound multiplicatively.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Create a report template with metric tables and key plots.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
prices = load_prices(['SPY', 'QQQ', 'AAPL'])
returns = prices.pct_change().dropna()
metrics = compute_risk_metrics(returns)
assert metrics['max_drawdown'] <= 0
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 04 Day 05): Explain Turnover and define every symbol clearly for an inflation surprise week with simultaneous equity and rate volatility.
   - Model answer: "I use Turnover as a decision bridge from market observations to position sizing. The formula is $TO_t=\frac{1}{2}\sum_i|w_{i,t}-w_{i,t-1}|$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then cap gross exposure when rolling 20-day volatility exceeds backtest 90th percentile. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Production question: Why does 'Capstone design and reporting' matter in live trading systems?
   - Model answer: "Capstone design and reporting matters because reliable notation and units prevent silent compounding and scaling errors in production PnL reporting. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and one production escalation rule.

### Interview Drill
- Prompt: "Walk me through Capstone design and reporting in a PM review after CPI surprise where benchmark drawdown accelerated."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
Which assumption in your analysis is weakest?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 04 Day 05 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [14]:
# Foundation demo: real prices + macro overlay with fallback
np.random.seed(405)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()
macro = load_fred_series('DGS10', start='2018-01-01').rename('dgs10')
joined = returns.join(macro, how='inner').dropna()

growth = float((1 + joined['SPY']).prod())
ann_return = float((1 + joined['SPY']).prod() ** (252 / joined.shape[0]) - 1)
ann_vol = float(joined['SPY'].std() * np.sqrt(252))
drawdown = float((prices['SPY'] / prices['SPY'].cummax() - 1).min())
rate_corr = float(joined['SPY'].corr(joined['dgs10']))

{
    'growth_multiple': growth,
    'annualized_return': ann_return,
    'annualized_volatility': ann_vol,
    'max_drawdown': drawdown,
    'spy_rate_corr': rate_corr,
}


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_46794/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'growth_multiple': 3.001916748261975,
 'annualized_return': 0.14223478242587606,
 'annualized_volatility': 0.19307148884826125,
 'max_drawdown': -0.33717259886422934,
 'spy_rate_corr': 0.008052798822480355}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [15]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(9405)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Capstone design and reporting',
    'week': 4,
    'day': 5,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Capstone design and reporting',
 'week': 4,
 'day': 5,
 'observe_annual_return': 0.14923384092579295,
 'observe_annual_vol': 0.20501282184020983,
 'reason_sharpe_proxy': 0.581592116315172,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 04 Day 05 Quiz

Topic: **Capstone design and reporting**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [16]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 109.000
price_t = 110.144
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages under inflation surprise week.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  cap exposure if rolling volatility breaches training 90th percentile.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Capstone design and reporting')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.010495)
print('  gross_return_expected  =', 1.010495)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages under inflation surprise week.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 109.0
  P_t    : 110.144
  r_t    : 0.010495 => 1.05%
  1+r_t  : 1.010495

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  cap exposure if rolling volatility breaches training 90th percentile.

Interview Question 3 (model answer):
  Topic: Capstone design and reporting
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Numeric verification:
  simple_return_expected = 0.010495
  gross_return_expected  = 1.010495


# Week 04 Day 06: Revision Sprint

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours across recall, rebuild, rerun, and retest blocks.

## Continuity and Handoff
- Previous checkpoint: Week 04 Day 05: Capstone design and reporting
- Previous lesson file: content/week-04/day-05.md
- Today's deliverable: Revision checklist with corrected errors and next-week retest priorities.
- Next handoff target: Week 04 Day 07: Portfolio Project
- Next lesson file: content/week-04/day-07.md

## Revision Plan
- 90 minutes: active recall of weekday concepts and manual formula rewrite.
- 90 minutes: rebuild one weekday code workflow from memory.
- 90 minutes: restart kernel and rerun at least two day notebooks end-to-end.
- 90 minutes: error-log triage, retest plan, and guardrail refinement.
- Optional 0-4 hours: deepen weak areas with extra interview drill and code hardening.

## Focus Areas
- Re-check all key metrics for one asset manually
- Stress-test one visualization with alternate scales
- Refine assumptions and caveats list

## Revision Output
- [ ] Updated error log entries
- [ ] Weak concepts re-tested
- [ ] Two notebooks rerun from fresh kernel
- [ ] Next-week risk list prepared


## Week 04 Day 06 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [17]:
# Revision sprint demo: rebuild weekly core diagnostics
np.random.seed(406)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()

summary = pd.DataFrame({
    'annual_return': returns.mean() * 252,
    'annual_vol': returns.std() * np.sqrt(252),
    'max_drawdown': (prices / prices.cummax() - 1).min(),
})
summary['sharpe_proxy'] = (summary['annual_return'] - 0.03) / summary['annual_vol'].replace(0, np.nan)
summary = summary.sort_values('sharpe_proxy', ascending=False)

print('Revision diagnostics (best risk-adjusted first):')
summary.round(4)


Revision diagnostics (best risk-adjusted first):


,annual_return,annual_vol,max_drawdown,sharpe_proxy
Ticker,,,,
AAPL,0.277000,0.306000,-0.385200,0.807300
QQQ,0.205600,0.238800,-0.351200,0.735400
SPY,0.151700,0.193100,-0.337200,0.630400


## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [18]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(9406)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Revision Sprint',
    'week': 4,
    'day': 6,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Revision Sprint',
 'week': 4,
 'day': 6,
 'observe_annual_return': 0.14923384092579295,
 'observe_annual_vol': 0.20501282184020983,
 'reason_sharpe_proxy': 0.581592116315172,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 04 Day 06 Quiz

Topic: **Revision Sprint**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [19]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 110.000
price_t = 111.210
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages under inflation surprise week.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  cap exposure if rolling volatility breaches training 90th percentile.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Revision Sprint')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.011000)
print('  gross_return_expected  =', 1.011000)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages under inflation surprise week.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 110.0
  P_t    : 111.21
  r_t    : 0.011 => 1.10%
  1+r_t  : 1.011

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  cap exposure if rolling volatility breaches training 90th percentile.

Interview Question 3 (model answer):
  Topic: Revision Sprint
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Numeric verification:
  simple_return_expected = 0.011
  gross_return_expected  = 1.011


# Week 04 Day 07: Portfolio Project

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours for implementation, validation, and communication drills.

## Continuity and Handoff
- Previous checkpoint: Week 04 Day 06: Revision Sprint
- Previous lesson file: content/week-04/day-06.md
- Today's deliverable: Capstone 1: Market risk dashboard
- Next handoff target: Week 05 Day 01: Supervised learning setup and leakage controls
- Next lesson file: content/week-05/day-01.md

## Project Title
Capstone 1: Market risk dashboard

## Problem Statement
Build an exploratory dashboard for five assets and produce a risk-focused memo.

## Data Sources
- Yahoo Finance
- FRED
- Optional benchmark index

## Implementation Steps
1. Define analysis universe
2. Compute return and risk metrics
3. Create visual analytics panels
4. Stress-test metric robustness
5. Write and present conclusions

## Evaluation Metrics
- Coverage breadth
- Risk interpretation quality
- Reproducibility
- Communication clarity

## Execution Standard
- [ ] Notebook/script runs from clean start without hidden state
- [ ] Outputs include at least one diagnostic table and one chart
- [ ] One explicit risk guardrail and fallback action are documented

## Deliverables
- Notebook or script output
- One-page summary memo
- Tracker update with completion and lessons learned


## Week 04 Day 07 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [20]:
# Project day demo: mini portfolio report with trade recommendation
np.random.seed(407)
assets = ['SPY', 'QQQ', 'TLT', 'GLD']
prices = load_market_prices(assets, start='2019-01-01')
returns = prices.pct_change().dropna()

annual_return = returns.mean() * 252
annual_vol = returns.std() * np.sqrt(252)
score = (annual_return - 0.03) / annual_vol.replace(0, np.nan)

report = pd.DataFrame({
    'annual_return': annual_return,
    'annual_vol': annual_vol,
    'sharpe_proxy': score,
}).sort_values('sharpe_proxy', ascending=False)

top_asset = report.index[0]
print('Project summary:')
print(report.round(4))
print(f"\nSuggested focus asset for follow-up research: {top_asset}")


Project summary:
        annual_return  annual_vol  sharpe_proxy
Ticker                                         
GLD          0.194100    0.172700      0.950000
QQQ          0.232200    0.240200      0.841900
SPY          0.177800    0.196000      0.754000
TLT         -0.005100    0.162600     -0.215600

Suggested focus asset for follow-up research: GLD


## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [21]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(9407)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Portfolio Project',
    'week': 4,
    'day': 7,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Portfolio Project',
 'week': 4,
 'day': 7,
 'observe_annual_return': 0.14923384092579295,
 'observe_annual_vol': 0.20501282184020983,
 'reason_sharpe_proxy': 0.581592116315172,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 04 Day 07 Quiz

Topic: **Portfolio Project**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [22]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 111.000
price_t = 112.277
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages under inflation surprise week.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  cap exposure if rolling volatility breaches training 90th percentile.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Portfolio Project')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.011505)
print('  gross_return_expected  =', 1.011505)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages under inflation surprise week.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 111.0
  P_t    : 112.277
  r_t    : 0.011505 => 1.15%
  1+r_t  : 1.011505

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  cap exposure if rolling volatility breaches training 90th percentile.

Interview Question 3 (model answer):
  Topic: Portfolio Project
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Numeric verification:
  simple_return_expected = 0.011505
  gross_return_expected  = 1.011505
